In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# This is how you define the MCP server used by the client
# Check out the docs on langchain_mcp_adapters: https://reference.langchain.com/python/langchain-mcp-adapters
client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http", # This can be stdio or streamable_http, stdio for local scripts, streamable_http for remote servers, see here information on transports: https://modelcontextprotocol.io/specification/2026-07-28/basic/transports
                                                # See here for docs in langchain: https://docs.langchain.com/oss/python/langchain/mcp#transports
                "url": "https://mcp.kiwi.com" # This is an MCP server that can be used to look up flights! It does not require an API key, so it is not super reliable (you may get some status 503s!), but still fun!
            }
    }
)

tools = await client.get_tools()

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "ollama:gemma4",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st 2027")]},
    config
    )

Error in post_writer
Traceback (most recent call last):
  File "/Users/alexlescionok/Desktop/code/lca-lc-foundations/.venv/lib/python3.13/site-packages/mcp/client/streamable_http.py", line 571, in post_writer
    await handle_request_async()
  File "/Users/alexlescionok/Desktop/code/lca-lc-foundations/.venv/lib/python3.13/site-packages/mcp/client/streamable_http.py", line 565, in handle_request_async
    await self._handle_post_request(ctx)
  File "/Users/alexlescionok/Desktop/code/lca-lc-foundations/.venv/lib/python3.13/site-packages/mcp/client/streamable_http.py", line 358, in _handle_post_request
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/alexlescionok/Desktop/code/lca-lc-foundations/.venv/lib/python3.13/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Server error '503 Service Unavailable' for url 'https://mcp.kiwi.com'
For more information che

BrokenResourceError: 

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='415849d2-2e7e-4704-bbbd-58755b8b882d'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-12T20:41:17.142164Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11854366875, 'load_duration': 3844213042, 'prompt_eval_count': 2294, 'prompt_eval_duration': 6617128000, 'eval_count': 40, 'eval_duration': 1369372000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a131d8-f804-7501-aefb-3f97311c5298-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2025'}, 'id': '7d436c0a-348f-46b5-a15c-186f6c9ebac3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2294, 'output_tokens': 40, 'total_tokens': 2334}),
              ToolMessage(content=[{'

In [ ]:
print(response["messages"][-1].content)

In [ ]:
# Local MCP server for looking up langchain docs!
client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "stdio", # This can be stdio or streamable_http, stdio for local scripts, streamable_http for remote servers, see here for docs: https://docs.langchain.com/oss/python/langchain/mcp#transports
                "url": "https://mcp.kiwi.com" # this is an MCP server that can look up flights.
            }
    }
)

tools = await client.get_tools()